# S=7 support sweep, part a of 2: Transformer AR vs causal encoder, 20,000 updates, sampled evaluation

Seven visible operation slots with `<identity>` fillers. The training support varies; the format, models and recipe
do not.

| condition | train k | held-out tests |
|---|---|---|
| S7_k02 | 0,2 | k=1 at every slot; k=3–7 |
| S7_k023 | 0,2,3 | k=1; k=4–7 |
| S7_k0234 | 0,2,3,4 | k=1; k=5–7 |
| S7_k02345 | 0,2,3,4,5 | k=1; k=6, 7 |
| S7_k023456 | 0,2,3,4,5,6 | k=1; k=7 |
| S7_k0234567 | 0,2,3,4,5,6,7 | k=1 only (k=7 fills every slot: trained zero-identity instructions) |

**Training.** 20,000 updates × 256 (5.12M examples).
* One common initialization per architecture: all conditions share capacity 7.
* Records at a common k are shared across conditions by index.

**Evaluation: the LAST checkpoint, on a SAMPLED test set** (about 50–100k records per run, instead of millions):
* Singletons: A/B/C × all 7 slots × 1000 grids = 21,000. Exhaustive.
* Each trained k: 5000 (word, grid) records.
* Each held-out longer k: 20,000 records. Words are cycled so each appears almost equally often.

The sizes are set in the config cell.

**This notebook = part a of 2: S7_k02, S7_k023, S7_k0234.** The other half is notebook 23b, and the two can run at
the same time on two runtimes. They share one results folder, but each condition has its own subfolder, so they do
not interfere.

**Send back `S7_Sweep_20k_parta_Review.ipynb`.** After BOTH halves finish, the optional last cell (in either
notebook) produces `S7_Sweep_20k_ALL_Review.ipynb` with all 12 runs.


In [ ]:
# --- 2. Bootstrap: locate or clone the PRIVATE repository (credentials are never printed or stored) ---
import base64, importlib.util, os, pathlib, subprocess, sys

REPO_URL = "https://github.com/zhexuandliu/Decompositionality.git"
REPO_BRANCH = "main"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

def _find_repo():
    candidates = [pathlib.Path("/content/Decompositionality"), pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    for p in candidates:
        if (p / "src" / "decompositionality").is_dir():
            return p.resolve()
    return None

REPO_DIR = _find_repo()
if REPO_DIR is None:
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    if not token:
        import getpass
        token = getpass.getpass("GitHub token with read access to the private repo (input hidden): ")
    # Pass the credential through environment-scoped git config (not the URL, not the command line,
    # not persisted in .git/config).
    _hdr = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    _env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1",
                GIT_CONFIG_KEY_0="http.https://github.com/.extraheader", GIT_CONFIG_VALUE_0=_hdr)
    REPO_DIR = pathlib.Path("/content/Decompositionality" if IN_COLAB else "Decompositionality").resolve()
    _r = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
                        env=_env, capture_output=True, text=True)
    del token, _hdr, _env
    if _r.returncode != 0:
        raise RuntimeError("git clone failed (check the GITHUB_TOKEN secret / repo access):\n" + _r.stderr[-2000:])

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for _mod in ("numpy", "pandas", "matplotlib", "torch"):
    if importlib.util.find_spec(_mod) is None:  # Colab already ships these; never reinstall torch/CUDA here
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _mod], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Repository:", REPO_DIR, "| commit:", REPO_COMMIT or "unknown")


In [ ]:
# --- CONFIGURATION -------------------------------------------------------------------------------------------------
MODE = os.environ.get("DECOMP_MODE", "full")
SEED = 42
RUN_CONDITIONS = ['S7_k02', 'S7_k023', 'S7_k0234']          # this half of the sweep (the other half runs in notebook 23b)
ALL_CONDITIONS = ["S7_k02", "S7_k023", "S7_k0234", "S7_k02345", "S7_k023456", "S7_k0234567"]
RUN_MODELS = ["transformer_ar", "causal_encoder"]
STEPS = 20000
SUBDIR = "S7_support_sweep_20k_v1"
EVAL_PLAN = {"n_single_grids": 1000, "n_seen_per_k": 5000, "n_long_per_k": 20000}
USE_DRIVE = IN_COLAB
RESULTS_ROOT = os.environ.get("DECOMP_RESULTS_ROOT") or (
    "/content/drive/MyDrive/decompositionality_results" if USE_DRIVE else str(REPO_DIR / "results"))
DEVICE = os.environ.get("DECOMP_DEVICE", "auto")
# -----------------------------------------------------------------------------------------------------------------
if USE_DRIVE and IN_COLAB and RESULTS_ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
import pandas as pd
from IPython.display import display
pd.set_option("display.max_rows", 500); pd.set_option("display.max_columns", 60); pd.set_option("display.width", 250)
from decompositionality import longk as LK, longk_exp as LX
_c = LX.make_cfg(RUN_CONDITIONS[0], RESULTS_ROOT, MODE, RUN_MODELS, SEED, SUBDIR,
                 **({"optimizer_steps": STEPS} if MODE == "full" else {}))
print("resolved recipe:", _c.train.semantic_dict())
print("results ->", LX.family_root(RESULTS_ROOT, MODE, SUBDIR))


In [ ]:
# --- Train (20,000 updates) + sampled LAST-checkpoint evaluation, sequentially (resumable) ---------------------------
status = LX.run_queue(RESULTS_ROOT, MODE, SEED, RUN_CONDITIONS, RUN_MODELS, repo_dir=str(REPO_DIR), device=DEVICE,
                      subdir=SUBDIR, steps=STEPS if MODE == "full" else None, sampled_plan=EVAL_PLAN)
display(status)


In [ ]:
# --- Report for THIS half + review export (reads saved artifacts only) ---------------------------------------------
rp = LX.report(RESULTS_ROOT, MODE, SEED, conditions=RUN_CONDITIONS, subdir=SUBDIR, out_name=f"seed_{SEED}_parta",
               title_note=f"Budget: {STEPS} updates; sampled evaluation {EVAL_PLAN}. Part a of 2.",
               review_name="S7_Sweep_20k_parta_Review.ipynb")
print("REVIEW FILE TO SEND BACK:", rp)
if IN_COLAB:
    from google.colab import files
    files.download(str(rp))


In [ ]:
# --- OPTIONAL: combined report for ALL 6 conditions (run after BOTH notebooks 23a and 23b have finished) ---------
rp_all = LX.report(RESULTS_ROOT, MODE, SEED, conditions=ALL_CONDITIONS, subdir=SUBDIR, out_name=f"seed_{SEED}_all",
                   title_note=f"Budget: {STEPS} updates; sampled evaluation {EVAL_PLAN}. All 12 runs.",
                   review_name="S7_Sweep_20k_ALL_Review.ipynb")
print("COMBINED REVIEW FILE:", rp_all)
if IN_COLAB:
    from google.colab import files
    files.download(str(rp_all))
